In [1]:
import os, sys, time, torch, dgl
from omegaconf import OmegaConf

# sys.path + imports de TON repo
repo_root = "../"  # <- adapte
if repo_root not in sys.path: sys.path.append(repo_root)
from python.create_dgl_dataset import TelemacDataset
from python.CustomMeshGraphNet import MeshGraphNet
from dgl.dataloading import GraphDataLoader

cfg = OmegaConf.create({
    "data_dir": "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Mesh8_base.bin",
    "dynamic_dir": ["/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_1_peak_1000_Group_1_peak_1000_0_0-80_interpolated.pkl", "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_1_peak_1200_Group_1_peak_1200_0_0-80_interpolated.pkl"],
    "ckpt_path": "./checkpoints/run_debug",
    "batch_size": 2,
    "lr": 1e-3,
    "epochs": 50,
    "num_input_features": 6+3,
    "num_edge_features": 3,
    "num_output_features": 3,
    "mp_layers": 10,
    "do_concat_trick": True,
    "num_processor_checkpoint_segments": 1,
})

def collate_fn(batch):
    graphs_at_t = [sequence[0] for sequence in batch]
    return dgl.batch(graphs_at_t)   # ici on retourne directement un DGLGraph batched

dataset = TelemacDataset(
    name="telemac_train",
    data_dir=cfg.data_dir,
    dynamic_data_files=cfg.dynamic_dir,
    split="train",
    ckpt_path=cfg.ckpt_path,
    normalize=True,
    sequence_length=1,
    overlap=0,
)
loader = GraphDataLoader(
    dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    drop_last=True,
    pin_memory=True,
    use_ddp=False,
    num_workers=0,
    collate_fn=collate_fn,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MeshGraphNet(cfg.num_input_features, cfg.num_edge_features, cfg.num_output_features,
                     processor_size=cfg.mp_layers,
                     hidden_dim_processor=64,
                     hidden_dim_node_encoder=64,
                     hidden_dim_edge_encoder=64,
                     hidden_dim_node_decoder=64,
                     do_concat_trick=cfg.do_concat_trick,
                     num_processor_checkpoint_segments=cfg.num_processor_checkpoint_segments).to(device)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

# smoke + mini-train
for epoch in range(cfg.epochs):
    t0 = time.time(); total = 0.0
    for g in loader:
        g = g.to(device)
        optimizer.zero_grad()
        pred = model(g.ndata["x"], g.edata["x"], g)
        loss = criterion(pred, g.ndata["y"])
        loss.backward()
        optimizer.step()
        total += loss.item()
    total /= len(loader)
    print(f"[epoch {epoch}] loss={total:.4e}  t={time.time()-t0:.2f}s")


Normalizing data...
[epoch 0] loss=1.1158e+00  t=2.89s
[epoch 1] loss=1.0563e+00  t=2.51s
[epoch 2] loss=1.0151e+00  t=2.63s
[epoch 3] loss=1.0004e+00  t=2.52s
[epoch 4] loss=1.0262e+00  t=2.51s
[epoch 5] loss=1.0065e+00  t=2.64s
[epoch 6] loss=9.9955e-01  t=2.51s
[epoch 7] loss=1.0146e+00  t=2.51s
[epoch 8] loss=9.9495e-01  t=2.64s
[epoch 9] loss=9.9390e-01  t=2.51s
[epoch 10] loss=1.0047e+00  t=2.51s
[epoch 11] loss=9.9737e-01  t=2.64s
[epoch 12] loss=1.0095e+00  t=2.53s
[epoch 13] loss=1.0049e+00  t=2.51s
[epoch 14] loss=1.0008e+00  t=2.51s
[epoch 15] loss=9.9589e-01  t=2.64s
[epoch 16] loss=9.9896e-01  t=2.52s
[epoch 17] loss=1.0066e+00  t=2.51s
[epoch 18] loss=9.9573e-01  t=2.64s
[epoch 19] loss=9.8807e-01  t=2.51s
[epoch 20] loss=1.0052e+00  t=2.51s
[epoch 21] loss=9.8943e-01  t=2.64s
[epoch 22] loss=9.9447e-01  t=2.51s
[epoch 23] loss=9.9751e-01  t=2.51s
[epoch 24] loss=9.9368e-01  t=2.64s
[epoch 25] loss=1.0010e+00  t=2.51s
[epoch 26] loss=9.9627e-01  t=2.51s
[epoch 27] loss=9.